## 0 - Imports


In [5]:
# MUST RUN THIS FIRST - Disable torch.compile BEFORE importing moshi
import torch
import torch._dynamo

# Completely disable torch compilation
torch._dynamo.config.suppress_errors = True
torch.set_float32_matmul_precision('high')

# Monkey-patch torch.compile to do nothing
original_compile = torch.compile
def no_compile(model, *args, **kwargs):
    print("torch.compile disabled - using eager mode")
    return model

torch.compile = no_compile

print("✓ Torch compilation disabled")

✓ Torch compilation disabled


In [6]:
import torch
import torchaudio
import numpy as np
import soundfile as sf
import librosa
import sounddevice as sd
import pandas as pd
import sys
import os
from pathlib import Path
import IPython.display as ipd

#fix filepathing
# sys.path.append(r"C:\Users\jking36\Documents\Master\Capstone\src\moshi\moshi")
#moshi imports
from moshi.models.loaders import CheckpointInfo
from moshi.models.tts import TTSModel, DEFAULT_DSM_TTS_REPO, DEFAULT_DSM_TTS_VOICE_REPO

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.10.0+cpu
CUDA available: False


In [7]:
print(sys.path)

print(os.getcwd())

['c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv\\python310.zip', 'c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv\\DLLs', 'c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv\\lib', 'c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv', '', 'c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv\\lib\\site-packages']
c:\Users\jking36\Documents\Master\Capstone


In [8]:
# Set device (use GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load the TTS model
print("Loading model...")
checkpoint_info = CheckpointInfo.from_hf_repo(DEFAULT_DSM_TTS_REPO)
tts_model = TTSModel.from_checkpoint_info(
    checkpoint_info,
    n_q=32,  # Number of codebook quantizers
    temp=0.6,  # Temperature for generation
    device=device,
    dtype=torch.float16 if device.type == 'cuda' else torch.float32
)

print("Model loaded successfully!")
print(f"See https://huggingface.co/{DEFAULT_DSM_TTS_VOICE_REPO} for available voices.")

Using device: cpu
Loading model...
Model loaded successfully!
See https://huggingface.co/kyutai/tts-voices for available voices.


In [31]:
# Text To Speech Demo - 45sec
text = "Testing the Moshi text to speech model, this is pretty cool!"
voice = "vctk/p228_023.wav"  # specify voice from huggingface

voice_path = tts_model.get_voice_path(voice)

print("Generating audio...")
audio_list = tts_model.simple_generate(
    text=text,
    voice=voice,
    cfg_coef=2.0
)

# It returns a list - get the first element
audio_tensor = audio_list[0]

# Convert to numpy
audio = audio_tensor.cpu().numpy()
if audio.ndim > 1:
    audio = audio[0]  # Get first channel if needed

audio = np.clip(audio, -1, 1)

print(f"✓ Generated {len(audio)/24000:.2f} seconds of audio")

# Play it
ipd.Audio(audio, rate=24000)

Generating audio...


Generating: 55it [00:39,  1.39it/s]


✓ Generated 4.24 seconds of audio


In [22]:
# Let's see what the TTSModel actually has
print("TTSModel methods:")
print([m for m in dir(tts_model) if not m.startswith('_') and callable(getattr(tts_model, m))])

# Also check if there's an LM model inside
print("\nTTSModel attributes:")
print([a for a in dir(tts_model) if not a.startswith('_')])

# Check if there's something like lm_gen or audio_lm
if hasattr(tts_model, 'lm_gen'):
    print(f"\nlm_gen type: {type(tts_model.lm_gen)}")
    print(f"lm_gen methods: {[m for m in dir(tts_model.lm_gen) if 'audio' in m.lower() or 'decode' in m.lower()]}")

TTSModel methods:
['from_checkpoint_info', 'generate', 'get_prefix', 'get_voice_path', 'lm', 'make_condition_attributes', 'mimi', 'prepare_script', 'simple_generate', 'warmup']

TTSModel attributes:
['cfg_coef', 'delay_steps', 'final_padding', 'from_checkpoint_info', 'generate', 'get_prefix', 'get_voice_path', 'lm', 'machine', 'make_condition_attributes', 'max_gen_length', 'max_speakers', 'mimi', 'multi_speaker', 'multistream', 'n_q', 'padding_bonus', 'prepare_script', 'simple_generate', 'temp', 'tokenizer', 'valid_cfg_conditionings', 'voice_repo', 'voice_suffix', 'warmup']


In [ ]:
# Save to file
import soundfile as sf

output_path = "output_tts.wav"
sf.write(output_path, audio, 24000)
print(f"Audio saved to {output_path}")

In [ ]:
# Try your own text here
custom_text = "Replace this with whatever you want to hear!"

# You can also try a different voice
# Browse https://huggingface.co/datasets/kyutai/tts-voices for options
custom_voice = "vctk/p225_001.wav"  # or try a different voice

# Generate
entries = tts_model.prepare_script([custom_text], padding_between=1)
voice_path = tts_model.get_voice_path(custom_voice)
condition_attributes = tts_model.make_condition_attributes([voice_path], cfg_coef=2.0)

pcms = []
def _on_frame(frame):
    if (frame != -1).all():
        pcm = tts_model.mimi.decode(frame[:, 1:, :]).cpu().numpy()
        pcms.append(np.clip(pcm[0, 0], -1, 1))

result = tts_model.generate([entries], condition_attributes, callback=_on_frame)
audio = np.concatenate(pcms)

# Play the audio
ipd.Audio(audio, rate=24000)